# Ejercicio final de la semana 1

Para demostrar que estás familiarizado con la API de OpenAI y también con Ollama, crea una herramienta que responda a una pregunta técnica
y la explique. ¡Esta es una herramienta que podrás usar durante el curso!

In [1]:
# imports

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [5]:
# constantes

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'


In [6]:
# Configura el entorno

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8]=='sk-proj-':
    print("La clave de API parece buena")
else:
    print("¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!")

openai = OpenAI()

La clave de API parece buena


In [7]:
# Aquí está la pregunta; escribe sobre esto para preguntar algo nuevo

system_prompt = "Necesito que actues como un programador experto en Python. \
Responde lo más detallado posible, dando ejemplos. \
Responde en formato markdown.\n"

user_prompt = """
Explica qué hace este código y por qué:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [9]:
# Haz que gpt-4o-mini responda, con streaming

def python_expert():
    # Llamada al modelo con streaming habilitado
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True  # Habilitar streaming
    )

    # Inicializar variables para mostrar el contenido progresivamente
    response = ""
    display_handle = display(Markdown(""), display_id=True)

    # Procesar cada fragmento de la respuesta
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

    return response


# Llamada a la función
final_response = python_expert()
print("Respuesta final completa:")
print(final_response)


El código que has proporcionado utiliza la expresión `yield from` junto con una comprensión de conjuntos en Python. Vamos a descomponerlo y explicarlo en detalle:

### Descomposición del código

python
yield from {book.get("author") for book in books if book.get("author")}


1. **Comprensión de conjuntos**: 
   - `{book.get("author") for book in books if book.get("author")}`
   - Este fragmento se llama "comprensión de conjunto" y se utiliza para crear un nuevo conjunto a partir de los elementos que pasan una determinada condición. 
   - En este caso, se itera sobre `books`, que se espera que sea una colección (como una lista o un conjunto) de diccionarios.
   - `book.get("author")` intenta obtener el valor asociado con la clave `"author"` de cada diccionario `book`.
   - `if book.get("author")` es una cláusula que filtra los libros, de modo que solo se incluyan aquellos que tienen un autor definido (es decir, que `get("author")` no devuelve `None` o una cadena vacía).
   - El resultado de la comprensión de conjuntos es un conjunto que contiene los autores de los libros que tienen un autor válido.

2. **`yield from`**:
   - La declaración `yield from` se utiliza en una función generadora para delegar parte de la operación de generación a otro generador, iterable o subgenerador.
   - En este caso, está enviando los elementos del conjunto generado (es decir, los autores) uno por uno desde el generador contenedor que lo implementa.

### Ejemplo

Imaginemos que tenemos la siguiente lista de libros:

python
books = [
    {"title": "Book One", "author": "Author A"},
    {"title": "Book Two", "author": "Author B"},
    {"title": "Book Three"},  # Este libro no tiene autor
    {"title": "Book Four", "author": ""},
    {"title": "Book Five", "author": "Author C"}
]


Si definimos una función generadora que utiliza el código que mencionaste:

python
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Uso de la función generadora
for author in get_authors(books):
    print(author)


### Salida del programa

La salida del programa sería:


Author A
Author B
Author C


### Resumen

- **Qué hace**: El código crea un conjunto de autores a partir de una lista de libros, ignorando aquellos libros que no tienen un autor. Luego, utiliza `yield from` para producir cada autor desde el generador.
- **Por qué**: Esto permite de manera eficiente iterar a través de los autores sin crear una lista adicional, y la utilización de un conjunto asegura que no haya duplicados en los autores devueltos.

### Beneficios

- **Eficiencia**: Usa `yield from` para generar valores solo cuando sea necesario, evitando cargar todo el conjunto en memoria a la vez.
- **Eliminación de duplicados**: La naturaleza del conjunto garantiza que no se repitan los autores.

Este uso de comprensiones de conjuntos y generadores es muy común en Python para realizar operaciones limpias y eficientes sobre colecciones de datos.

Respuesta final completa:
El código que has proporcionado utiliza la expresión `yield from` junto con una comprensión de conjuntos en Python. Vamos a descomponerlo y explicarlo en detalle:

### Descomposición del código

python
yield from {book.get("author") for book in books if book.get("author")}


1. **Comprensión de conjuntos**: 
   - `{book.get("author") for book in books if book.get("author")}`
   - Este fragmento se llama "comprensión de conjunto" y se utiliza para crear un nuevo conjunto a partir de los elementos que pasan una determinada condición. 
   - En este caso, se itera sobre `books`, que se espera que sea una colección (como una lista o un conjunto) de diccionarios.
   - `book.get("author")` intenta obtener el valor asociado con la clave `"author"` de cada diccionario `book`.
   - `if book.get("author")` es una cláusula que filtra los libros, de modo que solo se incluyan aquellos que tienen un autor definido (es decir, que `get("author")` no devuelve `None` o una caden

In [10]:
# Haz que Llama 3.2 responda

messages = [
    {
        "role": "user",
        "content": (
            "Necesito que actúes como un programador experto en Python. "
            "Responde lo más detallado posible, dando ejemplos. "
            "Responde en formato markdown. "
            "Explica qué hace este código y por qué: "
            "yield from {book.get('author') for book in books if book.get('author')}"
        )
    }
]

payload = {
    "model": MODEL_LLAMA,
    "messages": messages,
    "stream": False
}

response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)

# Procesar la respuesta JSON
response_json = response.json()

# Extraer el contenido de la respuesta
message_content = response_json.get('message', {}).get('content', 'No content received')

# Mostrar en formato Markdown
display(Markdown(message_content))

**Análisis del Código**

El código proporcionado es un ejemplo de cómo utilizar la función `yield from` en Python, junto con el método `.get()` y una estructura de control `for`.

```python
def obtenerAutoresLibros(books):
    yield from {book.get('author') for book in books if book.get('author')}
```

**Explicación del Código**

### **Función `.get()`**

La función `.get()` es un método que se utiliza para obtener una valor de un diccionario. Si el clave no existe en el diccionario, devuelve `None` por defecto.

```python
diccionario = {'clave': 'valor'}
valor = diccionario.get('clave')  # valor será 'valor'
```

En este caso, se utiliza `.get()` para obtener el valor de la clave `'author'` en cada libro del conjunto `books`. Si una libro no tiene la clave `'author'`, devuelve `None`.

### **Iterador `for`**

El iterador `for` se utiliza para iterar sobre un conjunto de objetos. En este caso, se utiliza para iterar sobre el conjunto de libros `books`.

```python
libros = [
    {'title': 'Libro 1', 'author': 'Autor 1'},
    {'title': 'Libro 2', 'author': None},
    {'title': 'Libro 3', 'author': 'Autor 3'}
]

for libro in libros:
    print(libro)
```

En este caso, se utiliza para iterar sobre cada libro en el conjunto `books` y obtener su contenido.

### **Función `.get()` con `yield from`**

La función `.get()` se combina con la función `.from()` para crear un generador que devuelve los valores de las claves `'author'` de los libros del conjunto `books`.

```python
def obtenerAutoresLibros(books):
    yield from {book.get('author') for book in books if book.get('author')}
```

En este caso, se utiliza `.get()` para obtener el valor de la clave `'author'` en cada libro. La función `yield from` es una forma de crear un generador que devuelve los valores del conjunto.

### **Ejemplo de Uso**

```python
libros = [
    {'title': 'Libro 1', 'author': 'Autor 1'},
    {'title': 'Libro 2', 'author': None},
    {'title': 'Libro 3', 'author': 'Autor 3'}
]

for autor in obtenerAutoresLibros(libros):
    print(autor)
```

En este caso, se imprime el siguiente resultado:

*   `Autor 1`
*   `None`
*   `Autor 3`

**Por qué funciona esto?**

La función `.get()` devuelve `None` si una clave no existe en un diccionario. En este caso, cuando se utiliza `.get()` para obtener la clave `'author'`, se devuelve `None` si una libro no tiene esa clave.

El iterador `for` se utiliza para iterar sobre el conjunto de libros. La función `.from()` es utilizada con el generador de los valores de las claves `'author'`. 

Este código permite utilizar un código más legible y enfocado en la lógica de negocio, mientras que el código subyacente es igual de eficiente.

**Conclusión**

El código proporcionado utiliza la función `.get()` para obtener los valores de las claves `'author'` de los libros del conjunto `books`. La función `.from()` se combina con un iterador `for` para crear un generador que devuelve estos valores. Esto permite utilizar una forma más legible y enfocada en la lógica de negocio, mientras que el código subyacente es igual de eficiente.